### Setup envs

In [1]:
import os
import logging
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

### Model definition

In [2]:
class UserModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.gender_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_genders'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_genders']) + 1, 4),
        ])

        self.lang_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_langs'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_langs']) + 1, 10),
        ])

        self.country_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_countries'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_countries']) + 1, 10),
        ])

        self.network_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_networks'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_networks']) + 1, 4),
        ])

        age_boundaries = np.array(conf['age_boundaries'])
        self.viewer_age_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.Discretization(age_boundaries.tolist()),
            tf.keras.layers.Embedding(len(age_boundaries), 2)
        ])

        self.centroids = tf.constant(conf['centroids'])
        self.viewer_lat_long_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.TextVectorization(
                standardize = None, split = self.classify,
                vocabulary = [str(i) for i in range(len(self.centroids))],
                max_tokens=len(self.centroids) + 2
                ),
            tf.keras.layers.Embedding(len(self.centroids) + 2, 2),
        ])

    @tf.function()
    def call(self, inputs):
        return tf.concat([
            self.gender_embedding(inputs["viewer_gender"]),
            self.lang_embedding(inputs["viewer_lang"]),
            self.country_embedding(inputs["viewer_country"]),
            self.network_embedding(inputs["viewer_network"]),
            self.viewer_age_embedding(inputs["viewer_age"]),
            self.viewer_lat_long_embedding(inputs["viewer_lat_long"]),
        ], axis = 1)

    @tf.keras.utils.register_keras_serializable()
    def classify(self, pair):
        """
        given a datapoint, compute the cluster closest to the datapoint. Return the cluster ID of that cluster.
        :param pair:
        :return: cluster ID
        """
        str_data = tf.strings.split(pair, sep = ",").values
        str_data = tf.map_fn(lambda x: tf.strings.regex_replace(x, "b'", ""), str_data)
        datapoints = tf.map_fn(lambda x: tf.strings.to_number(x), str_data, dtype = (tf.float32))
        datapoints = tf.reshape(datapoints, [-1, 2])

        expanded_centroids = tf.expand_dims(self.centroids, 1)
        expanded_vectors = tf.expand_dims(datapoints, 0)
        distances = tf.reduce_sum(tf.square(tf.subtract(expanded_vectors, expanded_centroids)), 2)
        clusters = tf.math.argmin(distances)
        return tf.strings.as_string(clusters)

In [3]:
class QueryModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		# We first use the user model for generating embeddings.
		self.embedding_model = UserModel(conf)
		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

In [4]:
class BroadcasterModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.broadcaster_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_broadcasters'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_broadcasters']) + 1, conf['broadcaster_embedding_dimension'])
        ])

    def call(self, broadcaster):
        return tf.concat([
            self.broadcaster_embedding(broadcaster),
        ], axis=1)

In [5]:
class CandidateModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		self.embedding_model = BroadcasterModel(conf)

		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dropout(0.5),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

### Load data

In [6]:
def load_data_file_cold(file, stats):
    print('loading file:' + file)
    training_df = pd.read_csv(
        file,
        skiprows=[0],
        names=["viewer",
               "broadcaster",
               "viewer_age",
               "viewer_gender",
               "viewer_longitude",
               "viewer_latitude",
               "viewer_lang",
               "viewer_country",
               "broadcaster_age",
               "broadcaster_gender",
               "broadcaster_longitude",
               "broadcaster_latitude",
               "broadcaster_lang",
               "broadcaster_country",
               "duration", 
               "viewer_network", 
               "broadcaster_network", 
               "count"], 
        dtype={
            'viewer': np.unicode,
            'broadcaster': np.unicode,
            'viewer_age': np.single,
            'viewer_gender': np.unicode,
            'viewer_longitude': np.single,
            'viewer_latitude': np.single,
            'viewer_lang': np.unicode,
            'viewer_country': np.unicode,
            'broadcaster_age': np.single,
            'broadcaster_longitude': np.single,
            'broadcaster_latitude': np.single,
            'broadcaster_lang': np.unicode,
            'broadcaster_country': np.unicode,
            'duration': np.single,
            'viewer_network': np.unicode,
            'broadcaster_network': np.unicode,
            'count': np.unicode,
        })

    values = {
        'viewer': 'unknown',
        'broadcaster': 'unknown',
        'viewer_age': 30,
        'viewer_gender': 'unknown',
        'viewer_longitude': 0,
        'viewer_latitude': 0,
        'viewer_lang': 'unknown',
        'viewer_country': 'unknown',
        'broadcaster_age': 30,
        'broadcaster_longitude': 0,
        'broadcaster_latitude': 0,
        'broadcaster_lang': 'unknown',
        'broadcaster_country': 'unknown',
        'duration': 13,
        'viewer_network': 'unknown',
        'broadcaster_network': 'unknown',
        "viewer_lat_long": tf.constant(["40.36393,-74.89611"]),
        'count': '1'
    }

    training_df = training_df.sample(frac=0.1)
    training_df.fillna(value=values, inplace=True)
    training_df['viewer_lat_long'] = training_df[['viewer_latitude', 'viewer_longitude']].apply(lambda x: '{},{}'.format(x[0],x[1]), axis=1)
    training_df['duration'] = np.log(1 + training_df['duration'])
    print(training_df.head(10))
    print(training_df.iloc[-10:])
    # stats.send_stats('data-size', len(training_df.index))
    return training_df


def load_training_data_cold(file, stats):
    ratings_df = load_data_file_cold(file, stats)
    print('creating data set')
    training_ds = (
        tf.data.Dataset.from_tensor_slices(
            ({
                "viewer": tf.cast(
                    ratings_df['viewer'].values,
                    tf.string),
                "viewer_gender": tf.cast(
                    ratings_df['viewer_gender'].values,
                    tf.string),
                "viewer_lang": tf.cast(
                    ratings_df['viewer_lang'].values,
                    tf.string),
                "viewer_country": tf.cast(
                    ratings_df['viewer_country'].values,
                    tf.string),
                "viewer_age": tf.cast(
                    ratings_df['viewer_age'].values,
                    tf.int32),
                "viewer_longitude": tf.cast(
                    ratings_df['viewer_longitude'].values,
                    tf.float16),
                "viewer_latitude": tf.cast(
                    ratings_df['viewer_latitude'].values,
                    tf.float16),
                "broadcaster": tf.cast(
                    ratings_df['broadcaster'].values,
                    tf.string),
                "viewer_network": tf.cast(
                    ratings_df['viewer_network'].values,
                    tf.string),
                "broadcaster_network": tf.cast(
                    ratings_df['broadcaster_network'].values,
                    tf.string),
                "duration": tf.cast(
                    ratings_df['duration'].values,
                    tf.float16),
                "viewer_lat_long": tf.cast(
                    ratings_df['viewer_lat_long'].values,
                    tf.string),
            })))

    return training_ds
            

def prepare_training_data_cold(train_ds):
    print('prepare_training_data')
    training_ds = train_ds.cache().map(lambda x: {
        "broadcaster": x["broadcaster"],
        "viewer": x["viewer"],
        "viewer_gender": x["viewer_gender"],
        "viewer_lang": x["viewer_lang"],
        "viewer_country": x["viewer_country"],
        "viewer_age": x["viewer_age"],
        "viewer_longitude": x["viewer_longitude"],
        "viewer_latitude": x["viewer_latitude"],
        "viewer_network": x["viewer_network"],
        "broadcaster_network": x["broadcaster_network"],
        "duration": x["duration"],
        "viewer_lat_long": x["viewer_lat_long"],
    }, num_parallel_calls=tf.data.AUTOTUNE,
       deterministic=False)

    print('done prepare_training_data')
    return training_ds

In [7]:
def get_broadcaster_data_set(train_ds):
    broadcasters = train_ds.cache().map(lambda x: x["broadcaster"], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    broadcasters_ds = tf.data.Dataset.from_tensor_slices(
        np.unique(list(broadcasters.as_numpy_iterator())))
    return broadcasters_ds

In [8]:
training_dataset = load_training_data_cold("csv/2022-01-12.csv", "")

loading file:csv/2022-01-12.csv
                                                   viewer  \
6143957   fa 94 9a 96 fe fa 5f 6c 74 ed 0f b4 b4 38 84 78   
3035573   82 d1 4e c8 c7 fd 51 90 4e 1e fb 9d df e1 b3 c8   
9478707   b7 4d b2 51 a9 e5 ad 36 d1 51 36 cb 70 37 01 62   
7639465   c0 93 af 19 a6 07 cf 05 02 c6 dd 27 a3 1b e7 4a   
10579963  64 31 1e db b0 c3 90 75 97 31 ac 19 c2 fd 66 e5   
2814306   d8 4c ee 8a c6 b1 5f 8c f3 56 7c 8a 18 4b 21 65   
8241075   dd d4 88 cc 6f d9 94 10 4d d8 c3 de 92 6e 20 9a   
10103952  e0 6e fa ce 69 85 b9 c3 d0 e7 48 68 88 4b 64 fc   
5759765   b0 b4 4a a3 21 40 70 43 ce 2d 3a 8a fd 1c 81 95   
4755534   a4 95 99 dd a5 ad ba 7a 90 38 52 eb 1d 99 cb b2   

                                              broadcaster  viewer_age  \
6143957   d2 d3 39 ca 99 12 81 83 72 98 9f 34 b7 2e c2 f2        32.0   
3035573   4f 6f 86 0a f8 52 8c d3 eb b3 c6 89 73 1e d1 87        34.0   
9478707   fa 8e 8c 73 99 b3 49 33 42 71 71 44 13 81 c8 44        28.0   
7639

In [9]:
train = prepare_training_data_cold(training_dataset)

prepare_training_data
done prepare_training_data


In [10]:
broadcasters_data_set = get_broadcaster_data_set(training_dataset)

### Prepare model conf

In [11]:
def get_list(training_data, key):
    return training_data.batch(1_000_000).map(lambda x: x[key], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)


def get_unique_list(data):
    return np.unique(np.concatenate(list(data)))

In [12]:
user_genders = get_list(train, 'viewer_gender')

In [13]:
user_langs = get_list(train, 'viewer_lang')

In [14]:
user_countries = get_list(train, 'viewer_country')

In [15]:
viewer_age = get_list(train, 'viewer_age')

In [16]:
user_networks = get_list(train, 'viewer_network')

### derive input dims

In [17]:
unique_user_genders = get_unique_list(user_genders)

In [18]:
len(unique_user_genders)

3

In [19]:
unique_user_langs = get_unique_list(user_langs)

In [20]:
len(unique_user_langs)

64

In [21]:
unique_user_countries = get_unique_list(user_countries)

In [22]:
len(unique_user_countries)

182

In [23]:
unique_user_networks = get_unique_list(user_networks)

In [24]:
len(unique_user_networks)

5

In [25]:
broadcaster_ids = get_list(train, 'broadcaster')

In [26]:
unique_broadcasters = get_unique_list(broadcaster_ids)

In [27]:
len(unique_broadcasters)

81392

In [28]:
broadcaster_embedding_dimension = 32

In [29]:
cold_start_conf = {
    'unique_genders': unique_user_genders,
    'unique_langs': unique_user_langs,
    'unique_countries': unique_user_countries,
    'unique_networks': unique_user_networks,
    'unique_broadcasters': unique_broadcasters,
    'broadcaster_embedding_dimension': broadcaster_embedding_dimension,
    'age_boundaries': [18, 25, 30, 35, 40, 45, 50, 55, 60, 65, float("inf")],
    'centroids': [[36.68147669256268, -82.8910274009993],
        [23.22243322909555, 78.23027450833709],
        [50.04997682638993, 0.22379313938744885],
        [37.9309447099281, -117.00741350764692],
        [-32.795864819917725, 148.7159172660312],
        [-18.570548393114084, -54.280255665692565],
        [13.921140442819565, 116.38740315555172],
        [29.78951080730802, 40.279515865947936]]
}

In [30]:
cold_start_conf

{'unique_genders': array([b'female', b'male', b'unknown'], dtype=object),
 'unique_langs': array([b'ar', b'az', b'bg', b'bn', b'bs', b'ca', b'co', b'da', b'de',
        b'el', b'en', b'es', b'et', b'eu', b'fa', b'fi', b'fr', b'ga',
        b'gl', b'gu', b'he', b'hi', b'hr', b'hu', b'hy', b'id', b'in',
        b'it', b'iw', b'ja', b'ko', b'lo', b'lt', b'lv', b'ml', b'mr',
        b'ms', b'my', b'nb', b'ne', b'nl', b'ny', b'pa', b'pl', b'ps',
        b'pt', b'ro', b'ru', b'si', b'sk', b'sl', b'sm', b'sq', b'sr',
        b'sv', b'ta', b'th', b'ti', b'tr', b'uk', b'ur', b'uz', b'vi',
        b'zh'], dtype=object),
 'unique_countries': array([b'150', b'419', b'AC', b'AD', b'AE', b'AF', b'AG', b'AI', b'AL',
        b'AO', b'AQ', b'AR', b'AS', b'AT', b'AU', b'AW', b'AX', b'AZ',
        b'BA', b'BB', b'BD', b'BE', b'BF', b'BG', b'BH', b'BJ', b'BN',
        b'BO', b'BQ', b'BR', b'BS', b'BT', b'BY', b'BZ', b'CA', b'CF',
        b'CH', b'CI', b'CL', b'CM', b'CN', b'CO', b'CR', b'CU', b'CV',
     

### query model

In [31]:
query_model = QueryModel(cold_start_conf)

### broadcaster model

In [32]:
candidate_model = CandidateModel(cold_start_conf)

### Candidate / Ranking model

In [33]:
class RankingModel(tf.keras.Model):

	def __init__(self):
		super().__init__()
		embedding_dimension = 32

		# Compute predictions.
		self.ratings = tf.keras.Sequential(
			[
				# Learn multiple dense layers.
				tf.keras.layers.Dense(256, activation = "relu"),
				tf.keras.layers.BatchNormalization(),           
				tf.keras.layers.Dense(64, activation = "relu"),
				tf.keras.layers.BatchNormalization(),   
				# Make rating predictions in the final layer.
				tf.keras.layers.Dense(1)
			]
		)

	def call(self, inputs):
		query_embeddings, positive_broadcaster_embeddings = inputs
		return self.ratings(tf.concat([query_embeddings, positive_broadcaster_embeddings], axis = 1))

In [34]:
ranking_model = RankingModel()

### Loss and metrics

In [35]:
ranking_task = tfrs.tasks.Ranking(
  loss = tf.keras.losses.MeanSquaredError(),
  metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

In [36]:
retrieval_task = tfrs.tasks.Retrieval(
    metrics=tfrs.metrics.FactorizedTopK(
        candidates=broadcasters_data_set.batch(128).map(candidate_model)
    )
)

In [37]:
from typing import Dict, Text

In [38]:
class TwoTowers(tfrs.models.Model) :

    def __init__(self, candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight):
        super().__init__()
        
        self.query_model: tf.keras.Model = query_model
        self.candidate_model: tf.keras.Model = candidate_model
        self.ranking_model: tf.keras.Model = ranking_model
        self.ranking_task = ranking_task
        self.retrieval_task = retrieval_task
        
        # The loss weights.
        self.ranking_weight = ranking_weight
        self.retrieval_weight = retrieval_weight
    
    def train_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        with tf.GradientTape() as tape:
            query_embeddings = self.query_model({
                "viewer_gender": features["viewer_gender"],
                "viewer_lang": features["viewer_lang"],
                "viewer_country": features["viewer_country"],
                "viewer_age": features["viewer_age"],
                "viewer_network": features["viewer_network"],
                "viewer_latitude": features["viewer_latitude"],
                "viewer_longitude": features["viewer_longitude"],
                "viewer_lat_long": features["viewer_lat_long"],
            })
            positive_broadcaster_embeddings = self.candidate_model(
                features["broadcaster"])
            
            labels = features["duration"]
            ranking_predictions = self.ranking_model(
                (query_embeddings, positive_broadcaster_embeddings)
            )
            ranking_loss = self.ranking_task(
                labels = labels,
                predictions = ranking_predictions,
            )
            
            retrieval_loss = self.retrieval_task(query_embeddings, positive_broadcaster_embeddings)
            
            regularization_loss = sum(self.losses)
            
            total_loss = regularization_loss + self.ranking_weight * ranking_loss + self.retrieval_weight * retrieval_loss
        
        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables)
        )
        
        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = ranking_loss 
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        
        return metrics

    def test_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        labels = features["duration"]
        
        query_embeddings = self.query_model({
            "viewer_gender": features["viewer_gender"],
            "viewer_lang": features["viewer_lang"],
            "viewer_country": features["viewer_country"],
            "viewer_age": features["viewer_age"],
            "viewer_network": features["viewer_network"],
            "viewer_latitude": features["viewer_latitude"],
            "viewer_longitude": features["viewer_longitude"],
            "viewer_lat_long": features["viewer_lat_long"],
        })
        positive_broadcaster_embeddings = self.candidate_model(
            features["broadcaster"])
        
        rating_predictions = self.ranking_model(
            (query_embeddings, positive_broadcaster_embeddings)
        )
        
        retrieval_loss = self.retrieval_task(query_embeddings, positive_broadcaster_embeddings)

        # The task computes the loss and the metrics.
        ranking_loss = self.ranking_task(labels = labels, predictions = rating_predictions)
        
        regularization_loss = sum(self.losses)
        
        total_loss = regularization_loss + self.ranking_weight * ranking_loss + self.retrieval_weight * retrieval_loss
        
        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = ranking_loss + retrieval_loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        return metrics        

In [39]:
ranking_weight = 1
retrieval_weight = 1

In [40]:
model = TwoTowers(candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight)

In [41]:
learning_rate = 0.1
batch_size = 16384
epochs = 50
patience = 3
top_k = 1999

In [42]:
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate))

In [43]:
tf.random.set_seed(42)
shuffled = train.shuffle(100_000, seed=42, reshuffle_each_iteration=True)

train_p80 = shuffled.take(80_000)
test_p20 = shuffled.skip(80_000).take(20_000)

cached_train = train_p80.shuffle(100_000).batch(batch_size)
cached_test = test_p20.batch(batch_size).cache()

In [44]:
# model.fit(train_ds, epochs=epochs)
callback = tf.keras.callbacks.EarlyStopping(monitor = "total_loss", 
    patience = patience,
    verbose=1,
    restore_best_weights=True)
hist = model.fit(
    cached_train,
    validation_data=cached_test,
    validation_freq=1,
    epochs=epochs,
    callbacks = [callback])

Epoch 1/30
Instructions for updating:
The `validate_indices` argument has no effect. Indices are always validated on CPU and never validated on GPU.
Instructions for updating:
Use fn_output_signature instead
5/5 [==============================] - 308s 64s/step - root_mean_squared_error: 4.0331 - factorized_top_k/top_1_categorical_accuracy: 0.0020 - factorized_top_k/top_5_categorical_accuracy: 0.0041 - factorized_top_k/top_10_categorical_accuracy: 0.0053 - factorized_top_k/top_50_categorical_accuracy: 0.0135 - factorized_top_k/top_100_categorical_accuracy: 0.0250 - loss: 15.8744 - regularization_loss: 0.0076 - total_loss: 160618.4948 - val_root_mean_squared_error: 2.6755 - val_factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - val_factorized_top_k/top_5_categorical_accuracy: 5.0000e-04 - val_factorized_top_k/top_10_categorical_accuracy: 0.0012 - val_factorized_top_k/top_50_categorical_accuracy: 0.0052 - val_factorized_top_k/top_100_categorical_accuracy: 0.0099 - val_loss: 29673.3

5/5 [==============================] - 177s 33s/step - root_mean_squared_error: 1.4312 - factorized_top_k/top_1_categorical_accuracy: 0.0019 - factorized_top_k/top_5_categorical_accuracy: 0.0085 - factorized_top_k/top_10_categorical_accuracy: 0.0139 - factorized_top_k/top_50_categorical_accuracy: 0.0489 - factorized_top_k/top_100_categorical_accuracy: 0.0753 - loss: 2.0469 - regularization_loss: 0.0129 - total_loss: 136033.1510 - val_root_mean_squared_error: 1.4292 - val_factorized_top_k/top_1_categorical_accuracy: 3.5000e-04 - val_factorized_top_k/top_5_categorical_accuracy: 0.0014 - val_factorized_top_k/top_10_categorical_accuracy: 0.0032 - val_factorized_top_k/top_50_categorical_accuracy: 0.0153 - val_factorized_top_k/top_100_categorical_accuracy: 0.0320 - val_loss: 26411.0488 - val_regularization_loss: 0.0130 - val_total_loss: 26411.0625
Epoch 19/30
5/5 [==============================] - 169s 35s/step - root_mean_squared_error: 1.4324 - factorized_top_k/top_1_categorical_accuracy: 

In [45]:
hist.history

{'root_mean_squared_error': [4.0330705642700195,
  2.644423723220825,
  1.5675147771835327,
  1.4681808948516846,
  1.4538800716400146,
  1.4566606283187866,
  1.461955189704895,
  1.444156289100647,
  1.4553236961364746,
  1.499876618385315,
  1.451816201210022,
  1.438153862953186,
  1.4583779573440552,
  1.4551644325256348,
  1.449979305267334,
  1.4840666055679321,
  1.4594956636428833,
  1.431214451789856,
  1.4324020147323608,
  1.4400317668914795,
  1.433392882347107,
  1.4327343702316284,
  1.4294573068618774,
  1.4322538375854492,
  1.4593087434768677,
  1.4433062076568604,
  1.4356192350387573,
  1.4275027513504028,
  1.4256235361099243,
  1.4504826068878174],
 'factorized_top_k/top_1_categorical_accuracy': [0.002025000052526593,
  0.0017750000115484,
  0.00047500000800937414,
  0.0005499999970197678,
  0.000750000006519258,
  0.000750000006519258,
  0.0006249999860301614,
  0.0005499999970197678,
  0.0012000000569969416,
  0.0007249999907799065,
  0.0006500000017695129,
  0.

In [46]:
accuracy = hist.history["factorized_top_k/top_100_categorical_accuracy"][-1]
print(f"Retrieval top-100 accuracy: {accuracy:.4f}.")

Retrieval top-100 accuracy: 0.1155.


In [47]:
root_mean_squared_error = hist.history["root_mean_squared_error"][-1]
print(f"Ranking RMSE: {root_mean_squared_error:.4f}.")

Ranking RMSE: 1.4505.
